# Route 2: barred dwarfs with MUSE data from the full DESI parent sample

Input: `products/<name>_barred_muse_candidates.csv` from
`scripts/02c_any_sample_bars_muse.py` (all MUSE-covered dwarfs of the
velocity-limited DESI table, no star-formation cut; PRIORITY 1 = barred by the
SGA catalogue and z < 0.012, 2 = barred at higher z, 3 = in the bar catalogue
but below the ellipticity threshold, 4 = other MUSE-covered dwarfs).

1. cutouts + geometry for the priority 1-3 candidates, vetting by eye
2. automatic isophote sweep of **all** MUSE-covered dwarfs to find bars the
   SGA catalogue cannot see (it only contains SGA galaxies), then vetting of
   the hits
3. the TW-ready list

Copy this notebook to a working name before running (e.g. `my_route2.ipynb`),
so `git pull` never conflicts with your saved outputs.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "bartw").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
from bartw.config import load_config, data_dir
from bartw.photometry import fetch_cutout, cutout_size_for, measure_geometry, plot_geometry
from bartw.geometry import inclination_from_axis_ratio, tw_suitable, pa_difference
cfg = load_config(); g = cfg["geometry"]; PIX = g["cutout_pixscale_arcsec"]
PROD = ROOT / "products"

NAME = "all_dwarfs"                                   # the --name you gave to scripts/02c
cand = pd.read_csv(PROD / f"{NAME}_barred_muse_candidates.csv")
print(len(cand), "MUSE-covered dwarfs;", cand.PRIORITY.value_counts().sort_index().to_dict())
display(cand[cand.PRIORITY <= 3][["TARGETID", "RA", "DEC", "Z", "LOGMSTAR", "INCL_DEG", "BAR_ELLIP", "BAR_MINUS_DISC_ELLIP",
                                  "N_MUSE_CUBES", "MUSE_EXPTIME_S", "MUSE_PUBLIC", "MUSE_PROG_IDS", "PRIORITY"]].round(3))

## 1. Priority 1-3: cutouts, geometry, vetting

The AGN/galaxy table has no size columns, so the galaxy size is taken from
the image itself: isophotes are fitted outwards until they reach the sky, the
disc PA/ellipticity are medians over 0.5-0.9 of the last real isophote (grey
band), and the bar is searched inside half of it. If the galaxy fills the
cutout (`EDGE_LIMITED`), a cutout twice as large is fetched automatically.
Red dashed ellipse = outer disc radius used, green = bar.

A photometric disc PA error above 5 deg no longer rejects a galaxy: it sets
`PA_NEEDS_KIN`, because the stellar velocity field of the MUSE cube gives the
disc PA to 2-3 deg and is what the TW measurement will use.

In [ ]:
import requests, io
from PIL import Image as PILImage

CUTOUT_ARCSEC = 120.0          # first-guess cutout side when the table has no size; doubled if the galaxy fills it

def r25_for(row):
    """Catalogue size if the table has one (SGA R25, 3 x Tractor half-light radius,
    Tractor D25/2), else None -> size measured from the image."""
    cands = []
    for c, f in (("sma_sb25", 1.0), ("shape_r", 3.0), ("d25_adopted_arcsec", 0.5)):
        if c in row and np.isfinite(row[c]) and row[c] > 0:
            cands.append(f * float(row[c]))
    return max(cands + [8.0]) if cands else None

def cutout_jpg(ra, dec, size_arcsec=60, layer="ls-dr10"):
    size = int(size_arcsec / PIX)
    url = f"https://www.legacysurvey.org/viewer/jpeg-cutout?ra={ra}&dec={dec}&layer={layer}&pixscale={PIX}&size={size}"
    r = requests.get(url, timeout=60); r.raise_for_status()
    return PILImage.open(io.BytesIO(r.content))

def viewer_link(ra, dec):
    return f"https://www.legacysurvey.org/viewer?ra={ra:.5f}&dec={dec:.5f}&layer=ls-dr10&zoom=14"

def show_grid(df, ncol=4, size_arcsec=60):
    n = len(df); nrow = max(1, int(np.ceil(n / ncol)))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.9 * nrow), squeeze=False)
    for ax, (_, r) in zip(axes.ravel(), df.iterrows()):
        try:
            ax.imshow(cutout_jpg(r.RA, r.DEC, size_arcsec), origin="upper")
            lm = r.LOGMSTAR if "LOGMSTAR" in r and np.isfinite(r.LOGMSTAR) else np.nan
            eb = f" e_bar={r.BAR_ELLIP:.2f}" if np.isfinite(r.get("BAR_ELLIP", np.nan)) else ""
            ax.set_title(f"{int(r.TARGETID)}  P{int(r.PRIORITY)}\nz={r.Z:.4f} logM={lm:.1f} i={r.INCL_DEG:.0f}°{eb}\n"
                         f"MUSE {r.MUSE_EXPTIME_S/60:.0f} min {'cube' if r.N_MUSE_CUBES > 0 else 'raw'}", fontsize=8)
        except Exception as exc:
            ax.text(0.5, 0.5, f"cutout failed\n{exc}"[:60], ha="center", va="center", fontsize=7, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    plt.tight_layout()

def geometry_for(row, show=True):
    r25 = r25_for(row)
    size = cutout_size_for(r25, PIX) if r25 else int(CUTOUT_ARCSEC / PIX)
    for attempt in range(2):
        img, hdr = fetch_cutout(row.RA, row.DEC, g["cutout_layer"], PIX, size)
        res = measure_geometry(img, PIX, r25)
        if not res.get("edge_limited", False):
            break
        size *= 2                                   # galaxy fills the cutout: refetch larger
    r_used = res["r25_pix"] * PIX if np.isfinite(res.get("r25_pix", np.nan)) else np.nan
    if show:
        plot_geometry(img, res, PIX, title=f"{int(row.TARGETID)}  r band  R_disc={r_used:.0f}\"  ({res['size_mode']})")
    return dict(TARGETID=int(row.TARGETID), R25_ARCSEC=r_used, SIZE_MODE=res["size_mode"], EDGE_LIMITED=res.get("edge_limited", False),
                PA_DISC=res["pa_disc"], PA_DISC_ERR=res["pa_disc_err"],
                EPS_DISC=res["eps_disc"], INCL_GEOM=float(inclination_from_axis_ratio(1 - res["eps_disc"], g["q0"])),
                PA_BAR=res["pa_bar"], A_BAR_ARCSEC=res["a_bar"], A_BAR_ERR_ARCSEC=res["a_bar_err"], EPS_BAR=res["eps_bar"],
                DPA_DEG=float(pa_difference(res["pa_bar"], res["pa_disc"])) if np.isfinite(res["pa_bar"]) else np.nan,
                GEOM_STATUS=res["status"])

def tw_table(df, geo):
    ok, dpa, why = tw_suitable(geo.INCL_GEOM, geo.PA_BAR, geo.PA_DISC, g)
    geo = geo.copy()
    geo["TW_SUITABLE"] = ok                                   # geometry only (inclination, bar-disc angle)
    geo["PA_NEEDS_KIN"] = geo.PA_DISC_ERR > 5.0                # photometric PA too uncertain: use the kinematic PA from MUSE
    geo["TW_REASON"] = np.where(geo.GEOM_STATUS != "ok", "no bar: " + geo.GEOM_STATUS.astype(str),
                                np.where(ok, np.where(geo.PA_NEEDS_KIN, "ok (needs kinematic PA)", "ok"), why))
    # a bar axis ratio below ~0.15 is not physical: usually an edge-on companion or a linear artefact
    geo["EPS_BAR_SUSPICIOUS"] = geo.EPS_BAR > 0.85
    geo.loc[geo.EPS_BAR_SUSPICIOUS & ok, "TW_REASON"] = geo.loc[geo.EPS_BAR_SUSPICIOUS & ok, "TW_REASON"] + " (eps_bar > 0.85: check image)"
    out = df.merge(geo, on="TARGETID")
    out["A_BAR_KPC"] = out.A_BAR_ARCSEC * out.KPC_PER_ARCSEC
    return out

top = cand[cand.PRIORITY <= 3].reset_index(drop=True)
display(HTML("<br>".join(f'{int(r.TARGETID)} (P{int(r.PRIORITY)}): <a href="{viewer_link(r.RA, r.DEC)}" target="_blank">{r.RA:.4f} {r.DEC:+.4f}</a>'
                         for _, r in top.iterrows())))
show_grid(top)

In [ ]:
geo = pd.DataFrame([geometry_for(r) for _, r in top.iterrows()])
top_geo = tw_table(top, geo)
print("TW-suitable:", int(top_geo.TW_SUITABLE.sum()), "/", len(top_geo)); print(top_geo.TW_REASON.value_counts().to_string())
display(top_geo[["TARGETID", "Z", "LOGMSTAR", "PRIORITY", "R25_ARCSEC", "EDGE_LIMITED", "INCL_DEG", "INCL_GEOM", "PA_DISC", "PA_DISC_ERR",
                 "PA_BAR", "DPA_DEG", "A_BAR_ARCSEC", "A_BAR_KPC", "EPS_BAR", "GEOM_STATUS", "TW_SUITABLE", "PA_NEEDS_KIN", "TW_REASON"]].round(2))

In [ ]:
# ---- by-eye verdicts for the priority 1-3 galaxies: 'y' bar, 'w' weak/uncertain, 'n' no bar, 'x' unusable
VET = {
    # 39627...: "y  # note",
}
vet_path = PROD / f"{NAME}_vetting.csv"
old = pd.read_csv(vet_path) if vet_path.exists() else pd.DataFrame(columns=["TARGETID", "VET", "VET_NOTE"])
rows = [dict(TARGETID=int(k), VET=str(v).split("#")[0].strip(), VET_NOTE=(str(v).split("#", 1)[1].strip() if "#" in str(v) else ""))
        for k, v in VET.items()]
vet = pd.concat([old[~old.TARGETID.isin([r["TARGETID"] for r in rows])], pd.DataFrame(rows)], ignore_index=True)
vet.to_csv(vet_path, index=False)
print(f"{len(vet)} verdicts saved:", vet.VET.value_counts().to_dict())

## 2. Sweep all MUSE-covered dwarfs for bars the catalogue cannot see

Only SGA galaxies (D25 > 20") are in the bar catalogue. The other MUSE-covered
dwarfs have never been checked. This runs the isophote analysis on every one
(~3 s each; a few hundred galaxies take 10-30 min) and keeps those with a
measured bar (`GEOM_STATUS == 'ok'`). Results are cached in
`products/<name>_sweep_geometry.csv`, so re-running only does new galaxies.

In [ ]:
sweep_path = PROD / f"{NAME}_sweep_geometry.csv"
done = pd.read_csv(sweep_path) if sweep_path.exists() else pd.DataFrame(columns=["TARGETID"])
todo = cand[~cand.TARGETID.isin(done.TARGETID)]
print(len(todo), "galaxies to measure,", len(done), "already done")
rows = list(done.to_dict("records"))
for k, (_, r) in enumerate(todo.iterrows(), 1):
    try:
        rows.append(geometry_for(r, show=False))
    except Exception as exc:
        rows.append(dict(TARGETID=int(r.TARGETID), GEOM_STATUS=f"failed: {exc}"[:80]))
    if k % 25 == 0:
        pd.DataFrame(rows).to_csv(sweep_path, index=False); print(f"  {k}/{len(todo)}")
sweep = pd.DataFrame(rows); sweep.to_csv(sweep_path, index=False)
print(sweep.GEOM_STATUS.value_counts().to_string())

In [ ]:
hits = tw_table(cand, sweep[sweep.GEOM_STATUS == "ok"])
hits = hits.sort_values(["TW_SUITABLE", "EPS_BAR"], ascending=[False, False]).reset_index(drop=True)
print(len(hits), "dwarfs with a measured bar;", int(hits.TW_SUITABLE.sum()), "TW-suitable by geometry")
display(hits[["TARGETID", "Z", "LOGMSTAR", "PRIORITY", "INCL_GEOM", "PA_DISC", "PA_DISC_ERR", "PA_BAR", "DPA_DEG",
              "A_BAR_ARCSEC", "A_BAR_KPC", "EPS_BAR", "MUSE_EXPTIME_S", "TW_SUITABLE", "PA_NEEDS_KIN", "TW_REASON"]].round(2))
display(HTML("<br>".join(f'{int(r.TARGETID)}: <a href="{viewer_link(r.RA, r.DEC)}" target="_blank">{r.RA:.4f} {r.DEC:+.4f}</a>'
                         for _, r in hits.iterrows())))
show_grid(hits.head(20))

In [ ]:
# profiles of the TW-suitable sweep hits, to confirm the bars by eye (and add them to VET above)
for _, r in hits[hits.TW_SUITABLE].head(12).iterrows():
    geometry_for(r, show=True)

## 3. TW-ready list

Union of the vetted (`y`/`w`) priority candidates and the confirmed sweep
hits, with the geometry needed by `scripts/06_run_tw.py`. Download their
cubes from the ESO archive (`MUSE_PROG_IDS`, `dp_id`s from the hits table of
notebook 01 section 2) and run `scripts/05_stellar_kinematics.py` on each.

In [ ]:
vet = pd.read_csv(PROD / f"{NAME}_vetting.csv") if (PROD / f"{NAME}_vetting.csv").exists() else pd.DataFrame(columns=["TARGETID", "VET"])
ready = pd.concat([top_geo[top_geo.TARGETID.isin(vet[vet.VET.isin(["y", "w"])].TARGETID)],
                   hits[hits.TW_SUITABLE]], ignore_index=True).drop_duplicates("TARGETID")
ready = ready.merge(vet[["TARGETID", "VET"]], on="TARGETID", how="left")
cols = ["TARGETID", "RA", "DEC", "Z", "KPC_PER_ARCSEC", "LOGMSTAR", "PA_DISC", "PA_DISC_ERR", "INCL_GEOM", "PA_BAR", "DPA_DEG",
        "A_BAR_ARCSEC", "A_BAR_ERR_ARCSEC", "A_BAR_KPC", "EPS_BAR", "N_MUSE_CUBES", "MUSE_EXPTIME_S", "MUSE_PUBLIC", "MUSE_PROG_IDS",
        "TW_SUITABLE", "PA_NEEDS_KIN", "TW_REASON", "VET"]
ready[[c for c in cols if c in ready]].round(3).to_csv(PROD / f"{NAME}_tw_ready.csv", index=False)
print(len(ready), "TW-ready candidates ->", f"products/{NAME}_tw_ready.csv")
display(ready[[c for c in cols if c in ready]].round(2))